# Prepare selected linear-probe models for XAI

This notebook reads the completed ChestMNIST linear-probe trial tables, selects each model's best saved configuration by validation `mean_auc`, retrains only that configuration, and leaves image-to-logits models ready for XAI. It does not run a grid search or save results.

In [7]:
import gc
import sys
from pathlib import Path

import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.notebook import tqdm

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from src.data import get_small_data
from src.model import get_dino_backbone, get_feature_linear_probe

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [8]:
# Keep this small while developing XAI. Use 1.0 only when full-data probes are needed.
data_fraction = 0.25
dataset_seed = 0
probe_batch_size = 512
normalize_features = True

model_configs = [
    {"run_name": "dinov2_small_224", "model_name": "facebook/dinov2-small", "image_size": 224, "batch_size": 32},
    {"run_name": "dinov2_large_224", "model_name": "facebook/dinov2-large", "image_size": 224, "batch_size": 16},
    {"run_name": "dinov3_large_224", "model_name": "facebook/dinov3-vitl16-pretrain-lvd1689m", "image_size": 224, "batch_size": 16},
    {"run_name": "stanford_dinov2_xray_224", "model_name": "StanfordAIMI/dinov2-base-xray-224", "image_size": 224, "batch_size": 16},
    {"run_name": "rad_dino_518", "model_name": "microsoft/rad-dino", "image_size": 518, "batch_size": 8},
]

# One model is enough while implementing XAI methods.
models_to_run = ["dinov2_small_224"]
# models_to_run = [config["run_name"] for config in model_configs]

results_root = project_root / "outputs" / "chestmnist" / "linear_probe"

In [9]:
def best_saved_config(run_name):
    trials_path = results_root / run_name / "trials.csv"
    if not trials_path.exists():
        raise FileNotFoundError(f"Missing completed linear-probe trials: {trials_path}")
    trials = pd.read_csv(trials_path)
    best = trials.sort_values(
        ["mean_auc", "f1_macro", "f1_micro", "mean_accuracy"],
        ascending=False,
    ).iloc[0]
    return {
        "lr": float(best["lr"]),
        "weight_decay": float(best["weight_decay"]),
        "epochs": int(best["epoch"]),
        "threshold": float(best["threshold"]),
        "saved_mean_auc": float(best["mean_auc"]),
    }

selected_configs = {run_name: best_saved_config(run_name) for run_name in models_to_run}
pd.DataFrame.from_dict(selected_configs, orient="index")

,lr,weight_decay,epochs,threshold,saved_mean_auc
dinov2_small_224,0.003,0.0,100,0.1,0.71918


In [10]:
def pool_backbone_output(outputs):
    features = getattr(outputs, "pooler_output", None)
    return features if features is not None else outputs.last_hidden_state[:, 0]


def extract_one_split(backbone, loader, progress, split_name):
    features, labels, ids = [], [], []
    backbone.eval()
    with torch.no_grad():
        for batch in loader:
            batch_features = pool_backbone_output(backbone(pixel_values=batch["images"].to(device)))
            if normalize_features:
                batch_features = F.normalize(batch_features, dim=1)
            features.append(batch_features.cpu())
            labels.append(batch["labels"].float().cpu())
            ids.extend(list(batch["ids"]))
            progress.update(1)
            progress.set_postfix(split=split_name)
    return torch.cat(features), torch.cat(labels), ids


def extract_features_for_model(backbone, train_loader, val_loader, run_name):
    with tqdm(total=len(train_loader) + len(val_loader), desc=f"{run_name} features", leave=True) as progress:
        train_features, train_labels, train_ids = extract_one_split(backbone, train_loader, progress, "train")
        val_features, val_labels, val_ids = extract_one_split(backbone, val_loader, progress, "val")
    return {
        "train_features": train_features,
        "train_labels": train_labels,
        "train_ids": train_ids,
        "val_features": val_features,
        "val_labels": val_labels,
        "val_ids": val_ids,
    }


def train_selected_head(feature_bank, selected, run_name, num_classes):
    torch.manual_seed(dataset_seed)
    head = get_feature_linear_probe(feature_bank["train_features"].shape[1], num_classes).to(device)
    optimizer = torch.optim.AdamW(head.parameters(), lr=selected["lr"], weight_decay=selected["weight_decay"])
    criterion = nn.BCEWithLogitsLoss()
    loader = DataLoader(
        TensorDataset(feature_bank["train_features"], feature_bank["train_labels"]),
        batch_size=probe_batch_size,
        shuffle=True,
    )

    for _ in tqdm(range(selected["epochs"]), desc=f"{run_name} epochs", leave=True):
        head.train()
        total_loss = 0.0
        sample_count = 0
        for feature_batch, label_batch in loader:
            feature_batch = feature_batch.to(device)
            label_batch = label_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(head(feature_batch), label_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * feature_batch.shape[0]
            sample_count += feature_batch.shape[0]
    return head


class XAILinearProbe(nn.Module):
    supports_gradcam = True
    supports_occlusion = True

    def __init__(self, backbone, classifier, normalize=True):
        super().__init__()
        self.backbone = backbone
        self.classifier = classifier
        self.normalize = normalize
        for parameter in self.backbone.parameters():
            parameter.requires_grad = False

    def forward(self, images):
        features = pool_backbone_output(self.backbone(pixel_values=images))
        if self.normalize:
            features = F.normalize(features, dim=1)
        return self.classifier(features)

In [11]:
linear_probe_models = {}
linear_probe_info = {}
xai_example_batches = {}

selected_model_configs = [config for config in model_configs if config["run_name"] in models_to_run]
for config in selected_model_configs:
    run_name = config["run_name"]
    selected = selected_configs[run_name]
    print("=" * 80)
    print(run_name, selected)

    train_loader, val_loader, class_names = get_small_data(
        batch_size=config["batch_size"],
        image_size=config["image_size"],
        data_fraction=data_fraction,
        seed=dataset_seed,
    )
    train_feature_loader = DataLoader(train_loader.dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
    val_feature_loader = DataLoader(val_loader.dataset, batch_size=config["batch_size"], shuffle=False, num_workers=0)
    xai_example_batches[run_name] = next(iter(val_feature_loader))

    backbone = get_dino_backbone(config["model_name"]).to(device)
    feature_bank = extract_features_for_model(backbone, train_feature_loader, val_feature_loader, run_name)
    feature_probe = train_selected_head(feature_bank, selected, run_name, len(class_names))

    image_model = XAILinearProbe(
        backbone=backbone.cpu(),
        classifier=feature_probe.classifier.cpu(),
        normalize=normalize_features,
    ).eval()
    linear_probe_models[run_name] = image_model
    linear_probe_info[run_name] = {**config, **selected, "class_names": class_names}

    del feature_probe, feature_bank, backbone
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("Ready models:", list(linear_probe_models))

dinov2_small_224 {'lr': 0.003, 'weight_decay': 0.0, 'epochs': 100, 'threshold': 0.1, 'saved_mean_auc': 0.7191803578597388}
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz
Using downloaded and verified file: C:\Users\vasko\.medmnist\chestmnist.npz


dinov2_small_224 features:   0%|          | 0/702 [00:00<?, ?it/s]

dinov2_small_224 epochs:   0%|          | 0/100 [00:00<?, ?it/s]

Ready models: ['dinov2_small_224']


In [12]:
# Select one prepared model and image for XAI development.
model_key = next(iter(linear_probe_models))
model = linear_probe_models[model_key].to(device).eval()
example_batch = xai_example_batches[model_key]
image = example_batch["images"][0]
labels = example_batch["labels"][0]
class_names = linear_probe_info[model_key]["class_names"]

with torch.no_grad():
    probabilities = torch.sigmoid(model(image.unsqueeze(0).to(device)))[0].cpu()

top_indices = probabilities.topk(3).indices.tolist()
print("Model:", model_key)
print("Image:", example_batch["ids"][0])
print("Top predictions:", [(class_names[index], float(probabilities[index])) for index in top_indices])

Model: dinov2_small_224
Image: val_4933
Top predictions: [('infiltration', 0.19346211850643158), ('nodule', 0.05810611695051193), ('atelectasis', 0.05471852794289589)]
